# 02 — RFM Aggregation (SQL & pandas)

**Goal:** collapse the cleaned transaction table (one row per line item) into one row per customer, with Recency, Frequency, and Monetary computed.

**Approach:** compute it two independent ways -- once with real SQL against a SQLite database (`sql/rfm_aggregation.sql`), once with pandas `groupby` -- and verify they agree. If a SQL query and a pandas pipeline built independently produce identical numbers, that's strong evidence the logic is actually correct, not just "ran without error."

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/cleaned_transactions.csv", parse_dates=["InvoiceDate"])
print(df.shape)
df.head(3)

(397884, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00


## 1. SQL route: load into SQLite, run the real `.sql` file

This is not a query re-typed for show -- it's the exact `sql/rfm_aggregation.sql` file read off disk and executed.

In [2]:
conn = sqlite3.connect(":memory:")
df.to_sql("transactions", conn, index=False, if_exists="replace")

with open("../sql/rfm_aggregation.sql") as f:
    query = f.read()

rfm_sql = pd.read_sql_query(query, conn)
conn.close()

print(rfm_sql.shape)
rfm_sql.head()

(4338, 4)


,CustomerID,Recency,Frequency,Monetary
0,14646,2,73,280206.02
1,18102,1,60,259657.30
2,17450,9,46,194550.79
3,16446,1,2,168472.50
4,14911,2,201,143825.06


## 2. pandas route: the same logic via `groupby`

Built independently of the SQL above (not derived from it), using the same rules: reference date = max date + 1 day, Frequency = distinct invoices, Monetary = summed revenue.

In [3]:
reference_date = df["InvoiceDate"].dt.normalize().max() + pd.Timedelta(days=1)

rfm_pandas = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.dt.normalize().max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
).reset_index()

rfm_pandas["Monetary"] = rfm_pandas["Monetary"].round(2)
rfm_pandas["CustomerID"] = rfm_pandas["CustomerID"].astype(str)

print(rfm_pandas.shape)
rfm_pandas.sort_values("Monetary", ascending=False).head()

(4338, 4)


,CustomerID,Recency,Frequency,Monetary
1689,14646,2,73,280206.02
4201,18102,1,60,259657.30
3728,17450,9,46,194550.79
3008,16446,1,2,168472.50
1879,14911,2,201,143825.06


## 3. Cross-check: do SQL and pandas agree?

In [4]:
rfm_sql["CustomerID"] = rfm_sql["CustomerID"].astype(str)

merged = rfm_sql.merge(rfm_pandas, on="CustomerID", suffixes=("_sql", "_pandas"))

recency_match = (merged["Recency_sql"] == merged["Recency_pandas"]).all()
frequency_match = (merged["Frequency_sql"] == merged["Frequency_pandas"]).all()
monetary_match = (merged["Monetary_sql"] - merged["Monetary_pandas"]).abs().max() < 0.01

print(f"Same number of customers: {len(rfm_sql) == len(rfm_pandas)} ({len(rfm_sql)} vs {len(rfm_pandas)})")
print(f"Recency matches exactly:   {recency_match}")
print(f"Frequency matches exactly: {frequency_match}")
print(f"Monetary matches (<1p):    {monetary_match}")

Same number of customers: True (4338 vs 4338)
Recency matches exactly:   True
Frequency matches exactly: True
Monetary matches (<1p):    True


## 4. Look at the result

Now we have exactly what K-Means needs: one row per customer, three behavioral numbers.

In [5]:
rfm = rfm_sql.copy()  # SQL and pandas agree -- either is fine to carry forward
print(f"{len(rfm):,} customers")
rfm.describe()

4,338 customers


,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000
mean,93.059474,4.272015,2054.266459
std,100.012264,7.697998,8989.230441
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,307.415000
50%,51.000000,2.000000,674.485000
75%,142.750000,5.000000,1661.740000
max,374.000000,209.000000,280206.020000


In [6]:
rfm.to_csv("../data/rfm.csv", index=False)
print("Saved to ../data/rfm.csv")

Saved to ../data/rfm.csv


## Takeaways

- `sql/rfm_aggregation.sql` isn't decorative -- it's an executable, verified query, cross-checked line-for-line against an independent pandas implementation.
- Frequency is *distinct orders*, not line items -- a deliberate choice, not the pandas/SQL default behavior.
- Recency uses a reference date of "last purchase in dataset + 1 day" to avoid a Recency=0 edge case.
- Output: `data/rfm.csv` -- 4,338 customers x (Recency, Frequency, Monetary).

**Next up (`03_elbow_method.ipynb`):** Recency, Frequency, and Monetary are on wildly different scales (days vs. order counts vs. pounds sterling) -- before clustering, we need to fix that with `StandardScaler`, then use the Elbow Method to decide how many clusters (K) actually makes sense for this data.